# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# Load environment variables from config.env
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

print(f"OPENAI_API_KEY set: {OPENAI_API_KEY is not None}")
print(f"TAVILY_API_KEY set: {TAVILY_API_KEY is not None}")

In [ ]:
# Environment variables already loaded above — verify they're available
assert OPENAI_API_KEY is not None, "OPENAI_API_KEY not set! Create a config.env file."
assert TAVILY_API_KEY is not None, "TAVILY_API_KEY not set! Create a config.env file."
print("API keys verified successfully.")

### VectorDB Instance

In [ ]:
# Create a persistent ChromaDB client
CHROMA_DB_PATH = "chromadb_udaplay"
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
print(f"ChromaDB client created at: {CHROMA_DB_PATH}")

### Collection

In [ ]:
# Create OpenAI embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL
)
print(f"Embedding function created: {type(embedding_fn).__name__}")

In [ ]:
# Create or get the collection
COLLECTION_NAME = "udaplay"

try:
    collection = chroma_client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn
    )
    print(f"Reusing existing collection '{COLLECTION_NAME}' ({collection.count()} documents)")
except Exception:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn
    )
    print(f"Created new collection '{COLLECTION_NAME}'")

### Add documents

In [ ]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )